# Tutorial 2: JitRL MVP — Retrieval-Augmented Generation with Logit Biasing

**The simplest strategy for continual learning: retrieve relevant context, bias logits toward retrieved vocabulary.**

Think of it like an **open-book exam** vs. **studying for a closed-book exam**:

| Approach | Analogy | How it works |
|----------|---------|-------------|
| Fine-tuning (TTT) | Studying — memorize the material | Update model weights with new knowledge |
| **JitRL MVP (RAG)** | **Open-book exam — look up answers** | **Store documents, retrieve relevant chunks at query time** |

Why does this work? Modern language models are excellent at reading comprehension. If you put the right information in the model's context window, it can answer questions about it — **no weight changes needed**.

**In this tutorial you will learn:**
1. How TF-IDF retrieval finds relevant document chunks
2. How logit biasing nudges the model toward retrieved vocabulary
3. How to use `JitRLMVPEngine` for instant document learning and querying
4. Why this approach achieves **zero forgetting** by design

## Concepts: TF-IDF Retrieval

TF-IDF (**T**erm **F**requency — **I**nverse **D**ocument **F**requency) is a classic information retrieval technique that scores how important a word is to a document within a corpus.

**TF (Term Frequency):** How often a word appears in a specific document chunk.
$$\text{TF}(t, d) = \frac{\text{count of } t \text{ in } d}{\text{total words in } d}$$

**IDF (Inverse Document Frequency):** How rare the word is across all chunks. Common words like "the" get low IDF; domain-specific words like "qubit" get high IDF.
$$\text{IDF}(t) = \log\frac{N}{1 + \text{docs containing } t}$$

**TF-IDF = TF x IDF** — words that are frequent in *this* chunk but rare overall get the highest scores.

**Retrieval:** Given a query, we compute its TF-IDF vector and find document chunks with the highest **cosine similarity**:
$$\text{similarity}(q, d) = \frac{q \cdot d}{\|q\| \|d\|}$$

Our `TFIDFRetriever` uses scikit-learn's `TfidfVectorizer` under the hood.

In [ ]:
# TF-IDF Demo: build intuition with simple sentences
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# A tiny corpus of document chunks
corpus = [
    "Quantum computing uses qubits to perform calculations.",
    "Oracle Cloud offers database services and infrastructure.",
    "Trapped-ion processors achieve high quantum volume.",
    "Machine learning models can detect anomalies in data.",
]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())}")
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}  (chunks x features)\n")

# Show the feature names and their scores for the first chunk
features = vectorizer.get_feature_names_out()
scores = tfidf_matrix[0].toarray().flatten()
nonzero = [(features[i], f"{scores[i]:.3f}") for i in scores.argsort()[::-1] if scores[i] > 0]
print("Chunk 0 TF-IDF scores (highest first):")
for word, score in nonzero:
    print(f"  {word:20s} {score}")

# Retrieve: which chunk best matches a query?
query = "quantum qubits processor"
query_vec = vectorizer.transform([query])
similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

print(f"\nQuery: '{query}'")
print("Cosine similarities:")
for i, (sim, chunk) in enumerate(zip(similarities, corpus)):
    marker = " <-- best match" if i == similarities.argmax() else ""
    print(f"  [{i}] {sim:.4f}  {chunk[:60]}{marker}")

## Concepts: Logit Biasing

Even with retrieved context in the prompt, the model may generate tokens that drift away from the document's vocabulary. **Logit biasing** is a gentle nudge: we add a small positive value to the logits of tokens that appear in the retrieved chunks.

**How it works:**
1. Tokenize all retrieved chunks
2. Count how often each token ID appears across chunks
3. Normalize counts to [0, 1] by dividing by the max count
4. Multiply by `bias_strength` to get the final bias vector

$$\text{bias}[t] = \text{bias\_strength} \times \frac{\text{count}(t)}{\max(\text{counts})}$$

This bias is **added to the logits** before softmax during generation. It does NOT change the model weights — it just makes the model slightly more likely to use words from the retrieved context.

The `LogitBiaser` class handles this computation.

In [ ]:
# Logit Bias Demo: visualize which tokens get boosted
from transformers import AutoTokenizer
from continual_learning.jitrl.mvp.logit_bias import LogitBiaser

# Load just the tokenizer (lightweight, no GPU needed)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B", trust_remote_code=True)
vocab_size = tokenizer.vocab_size
print(f"Tokenizer vocab size: {vocab_size:,}")

# Create a LogitBiaser
biaser = LogitBiaser(tokenizer=tokenizer, bias_strength=2.0)

# Compute bias from sample chunks
sample_chunks = [
    "Oracle Labs established its Quantum Computing Research Division in 2019.",
    "The RedShift processor achieved quantum volume 512 with 72 trapped-ion qubits.",
]
bias = biaser.compute_bias(sample_chunks, vocab_size)

# Show top-biased tokens
top_k = 20
top_indices = bias.argsort(descending=True)[:top_k]
print(f"\nTop {top_k} biased tokens (bias_strength=2.0):")
print(f"{'Token ID':>10}  {'Token':>20}  {'Bias':>8}")
print("-" * 44)
for idx in top_indices:
    tid = idx.item()
    token_str = tokenizer.decode([tid])
    print(f"{tid:>10}  {repr(token_str):>20}  {bias[tid]:.4f}")

# Stats
nonzero_count = (bias > 0).sum().item()
print(f"\nTokens with nonzero bias: {nonzero_count:,} / {vocab_size:,} ({100*nonzero_count/vocab_size:.1f}%)")

## Concepts: The Full Pipeline

The `JitRLMVPEngine` ties retrieval and logit biasing into a simple two-phase pipeline:

### Phase 1: Learn (Chunk + Index)
```
document text
    --> split into chunks (max_chunk_words=300)
    --> add to TFIDFRetriever index
    --> done! (no weight updates, ~0.002s)
```

### Phase 2: Query (Retrieve + Bias + Generate)
```
question
    --> TFIDFRetriever.retrieve(query, top_k=3)  -->  relevant chunks
    --> prepend chunks as context in prompt
    --> LogitBiaser.compute_bias(chunks)  -->  bias vector
    --> model.generate() with bias added to logits
    --> answer
```

Key constructor parameters:
- `top_k` (default 3): number of chunks to retrieve per query
- `bias_strength` (default 2.0): how strongly to boost retrieved-chunk tokens
- `max_chunk_words` (default 300): chunk size for document splitting
- `max_context_tokens` (default 1024): max tokens in the augmented prompt

In [ ]:
# Load Model & Create Engine
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from continual_learning.jitrl.mvp.engine import JitRLMVPEngine

MODEL_NAME = "Qwen/Qwen2.5-1.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    trust_remote_code=True
).to(device)
model.eval()
print(f"Model loaded: {MODEL_NAME} ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")

# Create the JitRL MVP engine
engine = JitRLMVPEngine(
    model=model,
    tokenizer=tokenizer,
    top_k=3,
    bias_strength=2.0,
    max_chunk_words=300,
)
print(f"Engine created: top_k={engine.top_k}, bias_strength={engine._biaser.bias_strength}")

In [ ]:
# Learn from Document
import time
from pathlib import Path

doc_path = Path("data/sample_document.txt")
document_text = doc_path.read_text()
print(f"Document: {doc_path.name}")
print(f"Length: {len(document_text):,} chars, {len(document_text.split()):,} words\n")
print(f"Preview: {document_text[:200]}...\n")

# Learn!
start = time.perf_counter()
result = engine.learn(document_text)
elapsed = time.perf_counter() - start

print(f"Learning complete in {elapsed:.4f}s")
print(f"  Chunks added: {result['chunks_added']}")
print(f"  Tokens processed (approx): {result['tokens_processed']:,}")
print(f"  Method: {result['method']}")
print(f"  Total chunks in retriever: {engine._retriever.num_chunks}")
print(f"  Total documents: {engine.num_documents}")

In [ ]:
# Query the Engine — ask questions about the learned document

questions = [
    "Who led the Quantum Computing Research Division at Oracle Labs?",
    "What is the RedShift processor and what did it achieve?",
    "What is the Lattice Shield protocol?",
]

for q in questions:
    print(f"Q: {q}")
    
    # Show retrieved chunks
    chunks = engine._retriever.retrieve(q, top_k=engine.top_k)
    print(f"  Retrieved {len(chunks)} chunks (showing first 80 chars each):")
    for i, chunk in enumerate(chunks):
        print(f"    [{i}] {chunk[:80]}...")
    
    # Generate answer
    answer = engine.generate(q, max_new_tokens=128)
    print(f"  A: {answer.strip()}")
    print()

## Why Zero Forgetting?

The key insight of JitRL MVP is that **no model weights are modified**. All "learned" knowledge lives in the TF-IDF index, completely separate from the model's parameters.

This means:
- **General knowledge is perfectly preserved** — the model can still answer questions it knew before
- **No catastrophic forgetting** — learning document A does not interfere with document B
- **Instant "unlearning"** — just call `engine.clear()` to remove all learned documents

The trade-off: since we never update weights, the model's ability to use learned knowledge is limited by:
1. The quality of TF-IDF retrieval (simple bag-of-words matching)
2. The model's in-context comprehension ability
3. The context window size

For deeper integration of new knowledge, see Tutorial 3 (TTT engine) which actually updates weights.

In [ ]:
# Test Forgetting: general knowledge should be unaffected by learning

general_questions = [
    "What is the capital of France?",
    "What is 2 + 2?",
    "Who wrote Romeo and Juliet?",
]

print("=" * 60)
print("AFTER learning the Oracle Labs quantum computing document:")
print("Testing general knowledge (should be perfectly preserved)")
print("=" * 60)

for q in general_questions:
    answer = engine.generate(q, max_new_tokens=64)
    print(f"\nQ: {q}")
    print(f"A: {answer.strip()}")

print("\n" + "=" * 60)
print("Forgetting = 0: weights were never modified.")
print("General knowledge is identical to the base model.")
print("=" * 60)

In [ ]:
# Multi-Document Learning: retrieval picks the correct chunks from the right document

# Clear previous state
engine.clear()
print(f"After clear: {engine.num_documents} documents, {engine._retriever.num_chunks} chunks\n")

# Document 1: quantum computing (original)
doc1 = Path("data/sample_document.txt").read_text()
r1 = engine.learn(doc1)
print(f"Learned doc 1 (Quantum Computing): {r1['chunks_added']} chunks")

# Document 2: a synthetic document about a completely different topic
doc2 = """The Mediterranean Diet and Longevity

Research conducted between 2015 and 2023 has consistently shown that the Mediterranean 
diet is associated with reduced cardiovascular disease risk. The diet emphasizes olive oil, 
fresh vegetables, whole grains, and fish while limiting red meat and processed foods.

Dr. Maria Gonzalez at the Barcelona Nutrition Institute published a landmark 10-year 
longitudinal study in 2022, tracking 12,000 participants. Her findings showed a 31% 
reduction in heart disease among strict adherents to the Mediterranean diet compared 
to a control group following a typical Western diet.

The key biomarkers affected include LDL cholesterol, triglycerides, and inflammatory 
markers such as C-reactive protein. Participants also showed improved gut microbiome 
diversity, which is increasingly linked to immune system function."""
r2 = engine.learn(doc2)
print(f"Learned doc 2 (Mediterranean Diet): {r2['chunks_added']} chunks")

# Document 3: another synthetic topic
doc3 = """Mars Colonization Timeline

SpaceX's Starship program aims to establish a permanent human settlement on Mars by 2040. 
The first uncrewed cargo missions are planned for 2028, carrying habitat modules, solar 
panels, and water extraction equipment. Each Starship can deliver approximately 100 metric 
tons to the Martian surface.

NASA's Artemis program serves as a stepping stone, with lunar base operations beginning 
in 2026 to test life support systems and in-situ resource utilization (ISRU) technology 
that will later be adapted for Mars. The key challenge remains radiation shielding during 
the 7-month transit, with current solutions involving water walls and polyethylene barriers."""
r3 = engine.learn(doc3)
print(f"Learned doc 3 (Mars Colonization): {r3['chunks_added']} chunks")

print(f"\nTotal: {engine.num_documents} documents, {engine._retriever.num_chunks} chunks\n")

# Test retrieval picks the right document
test_queries = [
    ("What did Dr. Chen's team develop?", "doc1 - quantum"),
    ("What did Dr. Gonzalez study?", "doc2 - diet"),
    ("When will cargo missions go to Mars?", "doc3 - Mars"),
]

for query, expected_doc in test_queries:
    chunks = engine._retriever.retrieve(query, top_k=1)
    print(f"Query: {query}")
    print(f"  Expected source: {expected_doc}")
    print(f"  Retrieved chunk: {chunks[0][:100]}...")
    answer = engine.generate(query, max_new_tokens=80)
    print(f"  Answer: {answer.strip()}")
    print()

## Tuning Parameters

The JitRL MVP engine has several parameters you can tune:

| Parameter | Default | Effect |
|-----------|---------|--------|
| `top_k` | 3 | Number of chunks retrieved per query. Higher = more context but slower and may dilute relevance |
| `bias_strength` | 2.0 | How strongly to boost tokens from retrieved chunks. 0 = no bias, higher = stronger nudge |
| `max_chunk_words` | 300 | Words per chunk. Smaller = more precise retrieval, larger = more context per chunk |
| `max_context_tokens` | 1024 | Max tokens in the augmented prompt. Increase for longer contexts |

**General guidelines:**
- For **precise factual QA**, use smaller chunks (100-200 words) and lower top_k (1-2)
- For **summarization or synthesis**, use larger chunks (300-500) and higher top_k (3-5)
- `bias_strength` of 1.0-3.0 works well in practice; too high can cause repetitive outputs

In [ ]:
# Parameter Sweep: compare different top_k values

# Reset engine with the quantum computing document
engine.clear()
engine.learn(Path("data/sample_document.txt").read_text())

test_question = "What is the RedShift processor and what speedup did it demonstrate?"

print(f"Question: {test_question}")
print("=" * 70)

for k in [1, 2, 3, 5]:
    # Temporarily override top_k
    original_top_k = engine.top_k
    engine.top_k = k
    
    start = time.perf_counter()
    answer = engine.generate(test_question, max_new_tokens=100)
    elapsed = time.perf_counter() - start
    
    chunks = engine._retriever.retrieve(test_question, top_k=k)
    total_chunk_words = sum(len(c.split()) for c in chunks)
    
    print(f"\ntop_k={k} | {len(chunks)} chunks ({total_chunk_words} words) | {elapsed:.2f}s")
    print(f"  Answer: {answer.strip()[:200]}")
    
    engine.top_k = original_top_k

print("\n" + "=" * 70)
print("Note: more chunks = more context, but diminishing returns past top_k=3.")

In [ ]:
# Bonus: Compare WITH vs WITHOUT logit biasing

test_q = "What encryption protocol did Oracle develop with NIST?"

# With bias (default)
engine._biaser.bias_strength = 2.0
answer_biased = engine.generate(test_q, max_new_tokens=100)

# Without bias (set strength to 0)
engine._biaser.bias_strength = 0.0
answer_unbiased = engine.generate(test_q, max_new_tokens=100)

# Restore default
engine._biaser.bias_strength = 2.0

print(f"Question: {test_q}\n")
print("WITH logit bias (strength=2.0):")
print(f"  {answer_biased.strip()[:300]}\n")
print("WITHOUT logit bias (strength=0.0):")
print(f"  {answer_unbiased.strip()[:300]}\n")
print("Logit biasing typically helps the model stay on-topic and use")
print("domain-specific vocabulary from the retrieved context.")

## Exercises

Try these on your own to deepen your understanding:

### 1. Chunk Size Experiment
- Create engines with `max_chunk_words` values of 50, 100, 300, and 500
- Learn the same document with each and compare retrieval quality
- How does chunk size affect the number of chunks and answer precision?

### 2. Logit Bias Ablation
- Compare answers at `bias_strength` values of 0, 1, 2, 5, and 10
- At what point does high bias start causing repetitive or incoherent output?
- Plot the bias distribution histogram for different strengths

### 3. Measure Retrieval Accuracy
- Write 10 question-answer pairs from the sample document
- For each question, check if the correct chunk is in the top-k retrieved
- Compute Recall@k for k=1, 3, 5

### 4. Compare with TTT
- After completing Tutorial 3 (TTT engine), compare the same questions answered by:
  - JitRL MVP (retrieval + logit bias, zero forgetting)
  - TTT (weight updates, potential forgetting)
- When does each approach shine?

### 5. Custom Documents
- Learn a document from your own domain (a paper, a manual, notes)
- How well does TF-IDF retrieval work on your domain's vocabulary?
- Does logit biasing help or hurt for your specific use case?